<h2> Code for analysing speedups of Greedy and Bruteforce methods, compared to the original framework </h2>

In [ ]:
import pandas as pd

with open('../../results/explain_time/final_result.pkl', "rb") as f:
    df = pd.read_pickle(f)

df.columns
display(df)

,dataset,method,trial,total_time,mem_before_mb,mem_after_mb,mem_delta_mb
0,syn1,CFExplainerNew,0,127.219208,792.406250,796.492188,4.085938
1,syn1,CFExplainerNew,1,127.126056,796.492188,796.492188,0.000000
2,syn1,CFExplainerNew,2,127.072089,796.492188,796.492188,0.000000
3,syn1,GreedyCFExplainer,0,12.549347,796.492188,797.597656,1.105469
4,syn1,GreedyCFExplainer,1,12.611344,797.597656,797.597656,0.000000
5,syn1,GreedyCFExplainer,2,12.601941,797.597656,797.597656,0.000000
6,syn1,BFCFExplainer,0,68.905707,797.597656,797.597656,0.000000
7,syn1,BFCFExplainer,1,69.073286,797.597656,797.597656,0.000000
8,syn1,BFCFExplainer,2,68.948702,797.597656,797.597656,0.000000
9,syn1,DenseSkip,0,328.129609,797.597656,801.179688,3.582031


In [ ]:
# average results over multiple trials
df = df.groupby(['dataset', 'method'])['total_time'].mean().reset_index()
display(df)

,dataset,method,total_time
0,syn1,BFCFExplainer,68.975898
1,syn1,CFExplainerNew,127.139117
2,syn1,DenseFull,685.307858
3,syn1,DenseSkip,329.925967
4,syn1,GreedyCFExplainer,12.587544
5,syn2,BFCFExplainer,184.652611
6,syn2,CFExplainerNew,327.755086
7,syn2,DenseFull,3883.872737
8,syn2,DenseSkip,2116.528006
9,syn2,GreedyCFExplainer,48.575483


In [21]:
def calculate_speedups(group):
    """Calculate speedups for all methods against specified baselines."""
    result = group.copy()

    # Get baseline times
    dense_skip_time = group.loc[group['method'] == 'DenseSkip', 'total_time']
    dense_full_time = group.loc[group['method'] == 'DenseFull', 'total_time']
    sparse_time = group.loc[group['method'] == 'CFExplainerNew', 'total_time']


    # Calculate speedups if baselines exist
    if not dense_skip_time.empty:
        skip_baseline = dense_skip_time.iloc[0]
        result['speedup_vs_DenseSkip'] = skip_baseline / group['total_time']

    if not dense_full_time.empty:
        full_baseline = dense_full_time.iloc[0]
        result['speedup_vs_DenseFull'] = full_baseline / group['total_time']

    if not sparse_time.empty:
        full_baseline = sparse_time.iloc[0]
        result['speedup_vs_CFExplainerNew'] = full_baseline / group['total_time']

    return result

# Apply to each dataset
df = df.groupby('dataset').apply(calculate_speedups).reset_index(drop=True)
df

/tmp/ipykernel_54830/466245169.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('dataset').apply(calculate_speedups).reset_index(drop=True)


,dataset,method,total_time,speedup_vs_DenseSkip,speedup_vs_DenseFull,speedup_vs_CFExplainerNew
0,syn1,BFCFExplainer,68.975898,4.783207,9.935468,1.843240
1,syn1,CFExplainerNew,127.139117,2.595000,5.390220,1.000000
2,syn1,DenseFull,685.307858,0.481427,1.000000,0.185521
3,syn1,DenseSkip,329.925967,1.000000,2.077156,0.385357
4,syn1,GreedyCFExplainer,12.587544,26.210511,54.443333,10.100391
5,syn2,BFCFExplainer,184.652611,11.462215,21.033403,1.774982
6,syn2,CFExplainerNew,327.755086,6.457651,11.849924,1.000000
7,syn2,DenseFull,3883.872737,0.544953,1.000000,0.084389
8,syn2,DenseSkip,2116.528006,1.000000,1.835021,0.154855
9,syn2,GreedyCFExplainer,48.575483,43.571940,79.955412,6.747336


In [16]:
df.loc[df.method.isin(['DenseFull', 'DenseSkip', 'CFExplainerNew']), ['method', 'total_time']]

,method,total_time
1,CFExplainerNew,127.139117
2,DenseFull,685.307858
3,DenseSkip,329.925967
6,CFExplainerNew,327.755086
7,DenseFull,3883.872737
8,DenseSkip,2116.528006
11,CFExplainerNew,69.318096
12,DenseFull,144.050124
13,DenseSkip,31.519772
16,CFExplainerNew,87.769049


In [22]:
df.loc[df.method.isin(['GreedyCFExplainer', 'BFCFExplainer'])]

,dataset,method,total_time,speedup_vs_DenseSkip,speedup_vs_DenseFull,speedup_vs_CFExplainerNew
0,syn1,BFCFExplainer,68.975898,4.783207,9.935468,1.843240
4,syn1,GreedyCFExplainer,12.587544,26.210511,54.443333,10.100391
5,syn2,BFCFExplainer,184.652611,11.462215,21.033403,1.774982
9,syn2,GreedyCFExplainer,48.575483,43.571940,79.955412,6.747336
10,syn4,BFCFExplainer,22.188342,1.420556,6.492154,3.124077
14,syn4,GreedyCFExplainer,1.123625,28.051862,128.201252,61.691489
15,syn5,BFCFExplainer,18.616157,1.217677,11.985187,4.714671
19,syn5,GreedyCFExplainer,1.717674,13.197187,129.895528,51.097626
